# RaiFlow — EU AI Act Compliance Walkthrough

This notebook walks through how RaiFlow enforces EU AI Act compliance at every stage of your AI project's CI/CD pipeline.

**What you'll learn:**
1. How to scaffold a compliance manifest with `raiflow init`
2. How to run static compliance checks with `raiflow check`
3. How to interpret check results (pass / fail / skipped)
4. How to use the live browser dashboard
5. How to integrate RaiFlow into GitHub Actions

**Articles covered:** EU AI Act Articles 9–14 (Risk Management, Data Governance, Technical Documentation, Record-Keeping, Transparency, Human Oversight)

---
**Prerequisites:** `pip install raiflow`

In [1]:
%pip install raiflow --quiet

Note: you may need to restart the kernel to use updated packages.


## 1. Scaffold your project with `raiflow init`

`raiflow init` scans your project for AI framework imports (LangChain, OpenAI, Ollama, LlamaIndex, Transformers, FastAPI, etc.), infers a risk level, and writes two files:
- `raiflow.yaml` — your compliance manifest
- `.github/workflows/rai-compliance.yml` — a ready-to-use GitHub Actions workflow

Run this once in your project root:

In [1]:
# Run in your project directory (not here — this is for illustration)
# !raiflow init

# For this notebook, we'll create a minimal raiflow.yaml inline
import yaml
from pathlib import Path

manifest_content = """
system_name: "my-rag-app"
risk_level: "high"
eu_ai_act_articles:
  - "Article 9"
  - "Article 10"
  - "Article 12"
  - "Article 13"
  - "Article 14"

model_metadata:
  name: "llama3"
  version: "3.1"
  disclosure_flag: true

risk_management:
  assessment_path: "docs/risk_assessment.md"

oversight:
  override_endpoints:
    - "/api/override"
    - "/api/halt"

logging:
  middleware_active: true

data_governance:
  dataset_path: null
  format: "csv"
  protected_attributes:
    - "gender"
    - "ethnicity"

banned_models: []

robustness:
  red_team_prompts_path: null
  toxicity_threshold: 0.7
"""

Path('demo_raiflow.yaml').write_text(manifest_content)
print('demo_raiflow.yaml written')

demo_raiflow.yaml written


## 2. Load the manifest

RaiFlow parses `raiflow.yaml` into a typed `RaiFlowManifest` object using Pydantic.

In [2]:
from raiflow.manifest import load_manifest

manifest = load_manifest('demo_raiflow.yaml')
print(f'System: {manifest.system_name}')
print(f'Risk level: {manifest.risk_level}')
print(f'Disclosure flag: {manifest.model_metadata.disclosure_flag}')
print(f'Logging active: {manifest.logging.middleware_active}')

System: my-rag-app
Risk level: high
Disclosure flag: True
Logging active: True


## 3. Run compliance checks programmatically

`CheckRunner` executes all static checks for a given pipeline stage and returns a list of `CheckResult` objects.

| Stage | Checks run |
|---|---|
| `pre-commit` | Banned Model Scan |
| `ci` | + Transparency, Risk Mgmt, Human Oversight, Logging, Bias Detection, Robustness |
| `pre-deploy` | Same as `ci` |
| `post-deploy` | Banned Model Scan, Transparency, Risk Mgmt, Human Oversight, Logging |

In [3]:
from raiflow.gate import CheckRunner

runner = CheckRunner(manifest)
results = runner.run('ci')

for r in results:
    icon = '✅' if r.status == 'pass' else ('⏭️' if r.status == 'skipped' else '❌')
    print(f'{icon} [{r.article_id}] {r.check_name} — score={r.score:.2f} threshold={r.threshold:.2f} ({r.status})')
    if r.status == 'fail':
        print(f'   Fix: {r.remediation_hint}')

✅ [Banned Models] Banned Model Scan — score=1.00 threshold=1.00 (pass)
✅ [Article 13] Transparency by Design — score=1.00 threshold=1.00 (pass)
❌ [Article 9] Risk Management Documentation — score=0.00 threshold=1.00 (fail)
   Fix: Create a risk assessment document and set risk_management.assessment_path in raiflow.yaml
❌ [Article 14] Human Oversight Endpoints — score=0.00 threshold=1.00 (fail)
   Fix: Pass --target <source_file> to check endpoint definitions
✅ [Article 12] Logging Middleware Active — score=1.00 threshold=1.00 (pass)
⏭️ [Article 10] Bias Detection — score=0.00 threshold=0.90 (skipped)
⏭️ [Article 10] Robustness/Toxicity Check — score=0.00 threshold=0.70 (skipped)


## 4. Build and inspect the compliance report

After a run, RaiFlow produces a structured JSON report — the same artifact written to `raiflow-report.json` by the CLI.

In [4]:
import json
from raiflow.reporter import build_report

report = build_report(stage='ci', manifest=manifest, checks=results)

print(f"Overall status : {report['overall_status']}")
print(f"Stage          : {report['stage']}")
print(f"Generated at   : {report['generated_at']}")
print(f"Schema version : {report['schema_version']}")
print()
print('Checks:')
for c in report['checks']:
    print(f"  {c['status'].upper():8s} {c['rule_id']:12s} {c['check_name']}")

Overall status : fail
Stage          : ci
Generated at   : 2026-04-16T15:34:31.941213+00:00
Schema version : 1.0

Checks:
  PASS     BAN-1        Banned Model Scan
  PASS     ART13-1      Transparency by Design
  FAIL     ART9-1       Risk Management Documentation
  FAIL     ART14-1      Human Oversight Endpoints
  PASS     ART12-1      Logging Middleware Active
  SKIPPED  ART10-3      Bias Detection
  SKIPPED  ART10-4      Robustness/Toxicity Check


## 5. Score and risk level

RaiFlow computes an **Overall Score** and maps it to a **Risk Level** — the same values shown in the dashboard ScoreBanner.

In [5]:
passing = sum(1 for r in results if r.status == 'pass')
failing = sum(1 for r in results if r.status == 'fail')
skipped = sum(1 for r in results if r.status == 'skipped')
non_skipped = passing + failing

overall_score = round((passing / non_skipped) * 100) if non_skipped > 0 else None

def risk_level(score):
    if score is None: return 'Unknown'
    if score == 100:  return 'Low'
    if score >= 75:   return 'Medium'
    if score >= 50:   return 'High'
    return 'Critical'

print(f'Passing  : {passing}')
print(f'Failing  : {failing}')
print(f'Skipped  : {skipped}')
print(f'Score    : {overall_score}%')
print(f'Risk     : {risk_level(overall_score)}')

Passing  : 3
Failing  : 2
Skipped  : 2
Score    : 60%
Risk     : High


## 6. Using the CLI

From your project directory, the same checks run via the CLI:

```bash
# Headless — prints results to terminal
raiflow check --stage ci --no-dashboard

# With live browser dashboard (non-CI only)
raiflow check --stage ci

# Explicit dashboard flag
raiflow check --stage ci --dashboard

# Dry run — always exits 0 (useful for reporting without blocking)
raiflow check --stage ci --dry-run --no-dashboard
```

The dashboard opens at `http://127.0.0.1:8000/` and lets you:
- Select a pipeline stage
- Click **Run Checks** to trigger a live run
- Watch results stream in via SSE
- Download the JSON report with the **⬇ Download Report** button

## 7. GitHub Actions integration

`raiflow init` drops `.github/workflows/rai-compliance.yml` into your repo. It defines four jobs:

| Job | Stage | Blocks on failure |
|---|---|---|
| `pre-commit-checks` | `pre-commit` | PR merge |
| `compliance-gate` | `ci` | Build |
| `build-and-sign` | — | Produces signed artifact manifest |
| `deploy-gate` | `pre-deploy` | Deployment |

```yaml
# .github/workflows/rai-compliance.yml (excerpt)
- name: Run compliance gate
  run: raiflow check --stage ci --no-dashboard
```

In CI environments (`CI=true`), the dashboard is automatically suppressed — no flags needed.

In [6]:
# Clean up demo file
Path('demo_raiflow.yaml').unlink(missing_ok=True)
print('Done.')

Done.
